# 🏠 House Price Predictor — Linear Regression
## California Housing Dataset
**Author:** Rose Sharma  
**Task:** Maincrafts Technology — AI/ML Task 1  
**Objective:** Build and evaluate a Linear Regression model using the ML workflow

---

## 📦 Step 1 — Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dataset
from sklearn.datasets import fetch_california_housing

# Preprocessing & splitting
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# Model
from sklearn.linear_model import LinearRegression

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Save model
import pickle
import os

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print(f'   pandas     : {pd.__version__}')
print(f'   numpy      : {np.__version__}')
print(f'   sklearn    : imported')

---
## 📂 Step 2 — Load Dataset

In [ ]:
# Load California Housing dataset
california = fetch_california_housing(as_frame=True)

# Create DataFrame
df = pd.concat([california.data, california.target.rename('MedHouseVal')], axis=1)

print('✅ Dataset loaded successfully!')
print(f'   Shape: {df.shape}')
print(f'   Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print(f'\n📋 Features:')
for col in california.feature_names:
    print(f'   • {col}')
print(f'   • MedHouseVal (Target — Median House Value in $100,000s)')

In [ ]:
# View first 5 rows
print('📊 First 5 rows of dataset:')
df.head()

In [ ]:
# Feature descriptions
feature_desc = {
    'MedInc':    'Median income in block group',
    'HouseAge':  'Median house age in block group',
    'AveRooms':  'Average number of rooms per household',
    'AveBedrms': 'Average number of bedrooms per household',
    'Population':'Block group population',
    'AveOccup':  'Average number of household members',
    'Latitude':  'Block group latitude',
    'Longitude': 'Block group longitude',
    'MedHouseVal': 'Median house value (TARGET — in $100,000s)'
}

print('📖 Feature Descriptions:')
for feat, desc in feature_desc.items():
    print(f'   {feat:15s} → {desc}')

---
## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('📈 Statistical Summary:')
df.describe().round(2)

In [ ]:
# Check missing values
print('🔍 Missing Values Check:')
missing = df.isnull().sum()
if missing.sum() == 0:
    print('   ✅ No missing values found! Dataset is clean.')
else:
    print(missing[missing > 0])

# Check data types
print('\n📋 Data Types:')
print(df.dtypes)

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable — Median House Value Distribution',
             fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(df['MedHouseVal'], bins=50, color='#1a3c6e', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribution of House Prices')
axes[0].set_xlabel('Median House Value ($100,000s)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['MedHouseVal'].mean(), color='red',
                linestyle='--', label=f'Mean: {df["MedHouseVal"].mean():.2f}')
axes[0].axvline(df['MedHouseVal'].median(), color='orange',
                linestyle='--', label=f'Median: {df["MedHouseVal"].median():.2f}')
axes[0].legend()

# Box plot
axes[1].boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#2980b9', alpha=0.7))
axes[1].set_title('Box Plot of House Prices')
axes[1].set_ylabel('Median House Value ($100,000s)')
axes[1].set_xticklabels(['MedHouseVal'])

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Observation: House prices are right-skewed with prices capped at $500,000 (value=5.0)')

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
axes = axes.flatten()

colors = ['#1a3c6e','#2980b9','#27ae60','#f39c12',
          '#e74c3c','#8e44ad','#16a085','#d35400','#2c3e50']

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color=colors[i], alpha=0.8, edgecolor='white')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr_matrix = df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    mask=mask,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

# Top correlations with target
print('\n📊 Feature Correlations with Target (MedHouseVal):')
target_corr = corr_matrix['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)
for feat, corr in target_corr.items():
    bar = '█' * int(abs(corr) * 20)
    direction = '+' if corr > 0 else '-'
    print(f'   {feat:12s} : {direction}{bar} ({corr:.3f})')

In [ ]:
# Scatter plots — top 4 features vs target
top_features = ['MedInc', 'AveRooms', 'HouseAge', 'Latitude']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Top Features vs Median House Value', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['MedHouseVal'],
                    alpha=0.2, s=5, color=colors[i])
    axes[i].set_xlabel(feat, fontsize=11)
    axes[i].set_ylabel('MedHouseVal', fontsize=11)
    axes[i].set_title(f'{feat} vs House Value\n(corr: {df[feat].corr(df["MedHouseVal"]):.3f})',
                       fontsize=11)

plt.tight_layout()
plt.savefig('scatter_features.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✅ Key insight: MedInc (Median Income) has strongest positive correlation with house prices!')

In [ ]:
# Geographic visualization
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseVal'],
    cmap='RdYlGn',
    alpha=0.4,
    s=df['Population']/100,
    vmin=0, vmax=5
)
plt.colorbar(scatter, label='Median House Value ($100,000s)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('California Housing Prices by Geographic Location\n(dot size = population)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('geographic_map.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Coastal areas (LA, SF) show highest house prices!')

---
## ✂️ Step 4 — Feature Selection & Data Splitting

In [ ]:
# Select features and target
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

print(f'✅ Features selected: {list(X.columns)}')
print(f'   X shape: {X.shape}')
print(f'   y shape: {y.shape}')

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'\n📊 Data Split:')
print(f'   Training set   : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Test set       : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Features scaled using StandardScaler')
print('   (Zero mean, unit variance — improves model convergence)')
print(f'\n   Train mean after scaling: {X_train_scaled.mean():.6f} (≈ 0)')
print(f'   Train std after scaling : {X_train_scaled.std():.6f}  (≈ 1)')

---
## 🤖 Step 5 — Train Linear Regression Model

In [ ]:
# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('✅ Linear Regression model trained!')
print(f'\n📊 Model Coefficients (Feature Importance):')
coef_df = pd.DataFrame({
    'Feature':     X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

for _, row in coef_df.iterrows():
    direction = '▲' if row['Coefficient'] > 0 else '▼'
    print(f'   {direction} {row["Feature"]:12s}: {row["Coefficient"]:+.4f}')

print(f'\n   Intercept: {model.intercept_:.4f}')

In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#27ae60' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'],
               color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('Linear Regression Feature Coefficients\n(Green = Positive impact, Red = Negative impact)',
             fontsize=13, fontweight='bold')

for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.02 if val >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 📏 Step 6 — Evaluate Model (MAE, RMSE, R²)

In [ ]:
# Predictions
import numpy as npfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scorey_pred_train = model.predict(X_train_scaled)
y_pred_test  = model.predict(X_test_scaled)

# Calculate metrics
mae_train  = mean_absolute_error(y_train, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r2_train   = r2_score(y_train, y_pred_train)

mae_test   = mean_absolute_error(y_test, y_pred_test)
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test    = r2_score(y_test, y_pred_test)

print('='*55)/nprint('  📊 MODEL EVALUATION RESULTS')/nprint('='*55)/nprint('  {:<20} {:>12} {:>12}'.format('Metric', 'Train', 'Test'))/nprint('-'*55)/nprint('  {:<20} {:>12.4f} {:>12.4f}'.format('MAE', mae_train, mae_test))/nprint('  {:<20} {:>12.4f} {:>12.4f}'.format('RMSE', rmse_train, rmse_test))/nprint('  {:<20} {:>12.4f} {:>12.4f}'.format('R² Score', r2_train, r2_test))/nprint('='*55)/nprint(f'\n  💡 Interpretation:')
print(f'  MAE  = {mae_test:.4f} → Average prediction error of ${mae_test*100000:,.0f}')
print(f'  RMSE = {rmse_test:.4f} → Root mean squared error of ${rmse_test*100000:,.0f}')
print(f'  R²   = {r2_test:.4f} → Model explains {r2_test*100:.1f}% of variance in house prices')

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model, scaler.fit_transform(X), y,
                             cv=5, scoring='r2')
print(f'📊 5-Fold Cross Validation R² Scores:')
for i, score in enumerate(cv_scores, 1):
    print(f'   Fold {i}: {score:.4f}')
print(f'\n   Mean R² : {cv_scores.mean():.4f}')
print(f'   Std Dev  : {cv_scores.std():.4f}')
print(f'   ✅ Consistent scores indicate stable model performance')

---
## 📊 Step 7 — Visualize Results

In [ ]:
# Actual vs Predicted scatter plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Linear Regression — Model Evaluation Plots',
             fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, alpha=0.4, s=10,
                color='#1a3c6e', label='Predictions')
axes[0].plot(
    [min(y_test), max(y_test)],
    [min(y_test), max(y_test)],
    color='red', linestyle='--', linewidth=2, label='Perfect prediction'
)
axes[0].set_xlabel('Actual Values', fontsize=12)
axes[0].set_ylabel('Predicted Values', fontsize=12)
axes[0].set_title(f'Actual vs Predicted\nR² = {r2_test:.4f}', fontsize=12)
axes[0].legend()
axes[0].text(0.05, 0.92, f'MAE  = {mae_test:.3f}\nRMSE = {rmse_test:.3f}\nR²   = {r2_test:.3f}',
             transform=axes[0].transAxes,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
             fontsize=10, verticalalignment='top')

# Plot 2: Residuals
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.4, s=10,
                color='#e74c3c', label='Residuals')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values', fontsize=12)
axes[1].set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
axes[1].set_title('Residual Plot\n(should be randomly scattered around 0)', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Residual distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Residual Analysis', fontsize=14, fontweight='bold')

# Histogram of residuals
axes[0].hist(residuals, bins=50, color='#2980b9', alpha=0.8, edgecolor='white')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].axvline(x=residuals.mean(), color='orange', linestyle='--',
                label=f'Mean: {residuals.mean():.4f}')
axes[0].set_xlabel('Residual Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Residuals')
axes[0].legend()

# Q-Q plot style
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals\n(points on line = normally distributed)')
axes[1].get_lines()[0].set(color='#1a3c6e', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color='red', linewidth=2)

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Metrics comparison bar chart
metrics = ['MAE', 'RMSE', 'R²']
train_vals = [mae_train, rmse_train, r2_train]
test_vals  = [mae_test,  rmse_test,  r2_test]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, train_vals, width,
               label='Train', color='#1a3c6e', alpha=0.85)
bars2 = ax.bar(x + width/2, test_vals, width,
               label='Test', color='#27ae60', alpha=0.85)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Metric')
ax.set_title('Train vs Test Performance Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 💾 Step 8 — Save Model

In [ ]:
# Save model and scaler using pickle
os.makedirs('model', exist_ok=True)

with open('model/linear_regression_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ Model saved successfully!')
print('   model/linear_regression_model.pkl')
print('   model/scaler.pkl')

# Verify by loading back
with open('model/linear_regression_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('model/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

test_pred = loaded_model.predict(loaded_scaler.transform(X_test))
print(f'\n✅ Model verified! Test R²: {r2_score(y_test, test_pred):.4f}')

---
## 🎯 Step 9 — Predict on New Input (Optional UI)

In [ ]:
def predict_house_price(MedInc, HouseAge, AveRooms, AveBedrms,
                         Population, AveOccup, Latitude, Longitude):
    """
    Predict median house value for given input features.
    Returns predicted value in $100,000s and actual dollars.
    """
    input_data = np.array([[MedInc, HouseAge, AveRooms, AveBedrms,
                             Population, AveOccup, Latitude, Longitude]])
    input_scaled = loaded_scaler.transform(input_data)
    prediction = loaded_model.predict(input_scaled)[0]
    return prediction


# Example prediction — typical California house
print('🏠 Example Prediction:')
print('   Input Features:')
sample = {
    'MedInc':     4.5,     # Median income ~ $45,000
    'HouseAge':   25.0,    # 25 years old
    'AveRooms':   5.0,     # 5 rooms average
    'AveBedrms':  1.1,     # ~1 bedroom per household
    'Population': 1500.0,  # Block population
    'AveOccup':   3.0,     # 3 people per household
    'Latitude':   34.0,    # Los Angeles area
    'Longitude':  -118.0
}
for k, v in sample.items():
    print(f'   {k:12s}: {v}')

pred = predict_house_price(**sample)
print(f'\n   🎯 Predicted House Value : {pred:.4f} ($100,000s)')
print(f'   💰 In actual dollars     : ${pred * 100000:,.0f}')

---
## 📋 Step 10 — Final Summary Report

In [ ]:
print('='*60)
print('  📋 FINAL MODEL REPORT — Linear Regression')
print('  California Housing Dataset | Maincrafts Technology Task 1')
print('='*60)
print(f'\n  📂 Dataset')
print(f'     Samples  : {len(df):,}')
print(f'     Features : {X.shape[1]}')
print(f'     Target   : Median House Value ($100,000s)')
print(f'\n  ✂️  Data Split')
print(f'     Train    : {len(X_train):,} samples (80%)')
print(f'     Test     : {len(X_test):,} samples (20%)')
print(f'     Scaling  : StandardScaler')
print(f'\n  🤖 Model')
print(f'     Algorithm: LinearRegression (sklearn)')
print(f'     Intercept: {model.intercept_:.4f}')
print(f'\n  📊 Performance Metrics')
print(f'     {'Metric':<10} {'Train':>10} {'Test':>10}')
print(f'     {'-'*32}')
print(f'     {'MAE':<10} {mae_train:>10.4f} {mae_test:>10.4f}')
print(f'     {'RMSE':<10} {rmse_train:>10.4f} {rmse_test:>10.4f}')
print(f'     {'R²':<10} {r2_train:>10.4f} {r2_test:>10.4f}')
print(f'\n  🔄 Cross Validation (5-fold)')
print(f'     Mean R²  : {cv_scores.mean():.4f}')
print(f'     Std Dev  : {cv_scores.std():.4f}')
print(f'\n  💡 Key Findings')
print(f'     1. MedInc is the strongest predictor of house prices')
print(f'     2. Coastal locations (SF, LA) command premium prices')
print(f'     3. Model explains {r2_test*100:.1f}% of price variance')
print(f'     4. Average prediction error: ${mae_test*100000:,.0f}')
print(f'\n  📈 Improvement Ideas')
print(f'     1. Try polynomial features for non-linear relationships')
print(f'     2. Apply log transformation to skewed features')
print(f'     3. Try Ridge/Lasso regularization')
print(f'     4. Use Random Forest or XGBoost for better accuracy')
print(f'\n  Author : Rose Sharma | CSE AI | Rungta College, Bhilai')
print('='*60)